In [1]:
import sys
import os

sys.path.append("/engram/nklab/algonauts/ethan/whole_brain_encoder")
os.chdir("/engram/nklab/algonauts/ethan/whole_brain_encoder")
import fetch
import torch
import numpy as np
from pathlib import Path
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt


In [2]:
import numpy as np
from tqdm import tqdm
from pathlib import Path
import json
from datetime import datetime

for subj_id in [1, 2, 5, 7][3:]:

    base_path = Path(
        f"/engram/nklab/algonauts/ethan/whole_brain_encoder/results_sims/schaefer/enc_1_3_5_7_run_1_2/imagenet/subj_{subj_id:02d}"
    )
    files = [p for p in base_path.iterdir() if p.suffix != ".h5"]

    area = "labeled_area"

    sims = {"ground_truth_sims": {"corr": []}}

    def to_numpy(x):
        return x if isinstance(x, np.ndarray) else x.numpy()
    
    img_paths = []

    # --- Load and concatenate correlation data ---
    for p in tqdm(files, desc=f"Loading subj_{subj_id:02d}"):
        d = np.load(p, allow_pickle=True).item()
        for sim_type in ["ground_truth_sims"]:
            sim_dict = d[sim_type][area]
            sims[sim_type]["corr"].append(to_numpy(sim_dict["corr"]))
            
        img_paths.extend(d["img_paths"])

    for sim_type in sims:
        for metric in sims[sim_type]:
            sims[sim_type][metric] = np.concatenate(sims[sim_type][metric], axis=1)

    print("----------- subj_id:", subj_id, "-----------")
    for sim_type in sims:
        for metric in sims[sim_type]:
            print(f"{sim_type} - {metric}: {sims[sim_type][metric].shape}")

    # --- Compute summary stats ---
    top_idxs = np.argmax(sims["ground_truth_sims"]["corr"], axis=-1)  # flatten all voxels × images
    image_paths_info = [(img_paths[idx]) for idx in top_idxs] 
    
    # --- Save to JSON ---
    out_path = f"/engram/nklab/pf2477/brain_decoding/brain_adapter/brain_encoder_corr/{subj_id}_imagenet_paths.txt"
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w") as f:
        for p in image_paths_info:
            f.write(f"{p}\n")

    print(f"✅ Saved {len(image_paths_info)} paths to: {out_path}")
    print(f"Sample img_path: {image_paths_info[0]}")

Loading subj_07: 100%|██████████| 1000/1000 [01:10<00:00, 14.12it/s]


----------- subj_id: 7 -----------
ground_truth_sims - corr: (515, 1281167)
✅ Saved 515 paths to: /engram/nklab/pf2477/brain_decoding/brain_adapter/brain_encoder_corr/7_imagenet_paths.txt
Sample img_path: /share/data/imagenet-pytorch/train/n03873416/n03873416_61703.JPEG


In [3]:
base_path = Path(
    "/engram/nklab/algonauts/ethan/whole_brain_encoder/results_sims/schaefer/enc_1_3_5_7_run_1_2/imagenet/subj_01"
)
files = [p for p in base_path.iterdir() if p.suffix != ".h5"]

area = "labeled_area"

sims = {
    "model_pred_sims": {"corr": [], "cos": []},
    "ground_truth_sims": {"corr": [], "cos": []},
}
img_paths = []


def to_numpy(x):
    return x if isinstance(x, np.ndarray) else x.numpy()


for p in tqdm(files):
    d = np.load(p, allow_pickle=True).item()
    # print(d.keys())

    for sim_type in ["model_pred_sims", "ground_truth_sims"]:
        sim_dict = d[sim_type][area]
        # print(sim_dict.keys())
        sims[sim_type]["corr"].append(to_numpy(sim_dict["corr"]))
        # sims[sim_type]["mse"].append(to_numpy(sim_dict["mse_sims"]))
        sims[sim_type]["cos"].append(to_numpy(sim_dict["cos"]))

    img_paths.extend(d["img_paths"])

for sim_type in sims:
    for metric in sims[sim_type]:
        sims[sim_type][metric] = np.concatenate(sims[sim_type][metric], axis=1)

print("Shapes:")
for sim_type in sims:
    for metric in sims[sim_type]:
        print(f"{sim_type} - {metric}: {sims[sim_type][metric].shape}")

print(f"Total img_paths: {len(img_paths)}")
print(f"Sample img_path: {img_paths[0]}")

100%|██████████| 1000/1000 [00:33<00:00, 29.48it/s]


Shapes:
model_pred_sims - corr: (1000, 1281167)
model_pred_sims - cos: (1000, 1281167)
ground_truth_sims - corr: (515, 1281167)
ground_truth_sims - cos: (515, 1281167)
Total img_paths: 1281167
Sample img_path: /share/data/imagenet-pytorch/train/n04505470/n04505470_1001.JPEG


In [ ]:
subj = 1
hemi = "lh"
imgs, _, _ = fetch.top_NSD_imgs(subj, hemi, split=["test"])
imgs = [Image.fromarray(img.img) for img in imgs]

Number of valid voxels:  163842
Number of parcels:  498


In [19]:
def _to_pil(x):
    if isinstance(x, Image.Image):
        return x
    # x might be a numpy array (H, W, C) or (H, W)
    return Image.fromarray(x)


def _open_path(p):
    # img_paths could contain bytes; handle both
    if isinstance(p, bytes):
        p = p.decode("utf-8")
    return Image.open(p)


# def plot_topk_for_all_sims(i, sims, img_paths, imgs, k=5):
#     """
#     For a given example index i, plot 1x(1+k) grids (GT + top-k) for:
#       - modelpred_sims.{corr,mse,cos}
#       - gt_sims.{corr,mse,cos}
#     """
#     gt_img = _to_pil(imgs[i])

#     pretty = {
#         "model_pred_sims": "Model→Brain sims",
#         "ground_truth_sims": "Ground-truth sims",
#         "corr": "Pearson r",
#         "mse": "MSE",
#         "cos": "Cosine",
#     }

#     # for sim_type in ["model_pred_sims", "ground_truth_sims"]:
#     for sim_type in ["model_pred_sims"]
#         # for metric in ["corr", "cos"]:
#         for metric in ["corr"]:
#             sims_mat = sims[sim_type][metric]  # shape: (N_examples, N_candidates)
#             scores = sims_mat[i]  # shape: (N_candidates,)

#             # higher=better ranking (if your MSE is *lower=better*, invert before building sims)
#             if metric == "mse":
#                 scores *= -1
#             top_idxs = np.argsort(scores)[::-1][:k]

#             # load top images
#             top_imgs = [_open_path(img_paths[idx]) for idx in top_idxs]

#             # figure
#             fig, axes = plt.subplots(1, 1 + k, figsize=(3 * (1 + k), 3))
#             if not isinstance(axes, np.ndarray):
#                 axes = np.array([axes])

#             # GT first
#             axes[0].imshow(gt_img)
#             axes[0].set_title("Ground Truth", fontsize=10)
#             axes[0].axis("off")

#             # then top-k
#             for j, idx in enumerate(top_idxs, start=1):
#                 axes[j].imshow(top_imgs[j - 1])
#                 axes[j].set_title(f"Top {j}\nscore={scores[idx]:.3f}", fontsize=9)
#                 axes[j].axis("off")

#             print(f"{pretty[sim_type]} · {pretty[metric]} (i={i})")
#             fig.suptitle(title, fontsize=12)
#             plt.tight_layout()
#             plt.show()

def preprocess_center_crop(img: Image.Image, resize_size=256, crop_size=224) -> Image.Image:
    """Resize shortest side to `resize_size`, then center crop to `crop_size`."""
    # resize while keeping aspect ratio
    w, h = img.size
    if w < h:
        new_w, new_h = resize_size, int(h * resize_size / w)
    else:
        new_w, new_h = int(w * resize_size / h), resize_size
    img = img.resize((new_w, new_h), Image.BICUBIC)

    # center crop
    left = (new_w - crop_size) // 2
    top = (new_h - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size
    img = img.crop((left, top, right, bottom))

    return img


def plot_topk_for_all_sims(subj, i, sims, img_paths, imgs, k=5):
    """
    For a given example index i, plot 1x(1+k) grids (GT + top-k) for:
      - modelpred_sims.{corr,mse,cos}
      - gt_sims.{corr,mse,cos}
    """
    gt_img = _to_pil(imgs[i])

    pretty = {
        "model_pred_sims": "Model→Brain sims",
        "ground_truth_sims": "Ground-truth sims",
        "corr": "Pearson r",
        "mse": "MSE",
        "cos": "Cosine",
    }

    sim_type = "model_pred_sims"
    metric = "corr"

    sims_mat = sims[sim_type][metric]  # shape: (N_examples, N_candidates)
    scores = sims_mat[i]               # shape: (N_candidates,)

    # higher = better ranking (invert if needed)
    if metric == "mse":
        scores *= -1

    # top-1 index and image
    top_idx = np.argmax(scores)
    top_img = _open_path(img_paths[top_idx])

    top_img = preprocess_center_crop(top_img, resize_size=256, crop_size=224)

    root = "/engram/nklab/pf2477/brain_decoding/figures/imagenet_retrieval"
    os.makedirs(root, exist_ok=True)
    save_path = os.path.join(root, f"{subj}_{i}.png")
    top_img.save(save_path)

    # # figure (just one image)
    # plt.imshow(top_img)
    # print(f"Top 1\nscore={scores[top_idx]:.3f}")
    # plt.axis("off")

    # print(f"{pretty[sim_type]} · {pretty[metric]} (i={i})")
    # plt.show()

    # plt.imshow(gt_img)
    # print("Ground Truth")
    # plt.axis("off")
    # plt.show()


subjects = [1,2,5,7]
ids = [5, 6, 11, 19, 27, 38, 62, 60, 0, 13, 22, 31, 50, 57, 69, 70, 45, 61,75,81,82,95,103,105]
subj = 1
hemi = "lh"
imgs, _, _ = fetch.top_NSD_imgs(subj, hemi, split=["test"])
imgs = [Image.fromarray(img.img) for img in imgs]

for subj in subjects[1:]:
    for i in ids:
        plot_topk_for_all_sims(subj, i, sims, img_paths, imgs, k=1)


Number of valid voxels:  163842
Number of parcels:  498
